In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.max_columns', 60)

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (average_precision_score, f1_score,
                             balanced_accuracy_score, roc_auc_score,
                             matthews_corrcoef)

# Load ProstT5 embeddings
prot_data = np.load("../data/features_prostt5_disorder.npz", allow_pickle=True)
prot_X_all = prot_data['X']
prot_accs = list(prot_data['accs'])
prot_idx = {a: i for i, a in enumerate(prot_accs)}

# Load master + GO Slim
master = pd.read_csv("../data/master_clean.csv")
keep = master['acc'].isin(prot_accs) & master['cluster'].notna()
master = master[keep].reset_index(drop=True)
y = master['d2o'].values.astype(int)
groups = master['cluster'].astype(int).values

bp = pd.read_csv("../data/features_GO_BP_slim.csv", index_col=0)
mf = pd.read_csv("../data/features_GO_MF_slim.csv", index_col=0)
cc = pd.read_csv("../data/features_GO_CC_slim.csv", index_col=0)

def make_X_prostt5(all_go):
    X_seq = np.stack([prot_X_all[prot_idx[a]] for a in master['acc']])
    if not all_go:
        return X_seq.astype(np.float32)
    parts = [X_seq,
             bp.reindex(master['acc']).fillna(0).values,
             mf.reindex(master['acc']).fillna(0).values,
             cc.reindex(master['acc']).fillna(0).values]
    return np.hstack(parts).astype(np.float32)

# Same shared splits, same threshold, same RF hyperparameters as W7
cv = GroupKFold(n_splits=5)
splits = list(cv.split(np.zeros(len(master)), y, groups))
THRESHOLD = float(y.mean())
print(f"n={len(master)}, positives={y.sum()}, clusters={len(np.unique(groups))}, "
      f"threshold={THRESHOLD:.3f}")

results = []
for cond_id, cond_label, all_go in [(1, "Seq only (ProstT5)", False),
                                    (8, "Seq + all GO (ProstT5)", True)]:
    X = make_X_prostt5(all_go)
    print(f"\n=== Condition {cond_id}: {cond_label} (X shape {X.shape}) ===")
    for fold, (tr, te) in enumerate(splits):
        clf = RandomForestClassifier(
            n_estimators=500, class_weight='balanced',
            min_samples_leaf=3, n_jobs=-1, random_state=42)
        clf.fit(X[tr], y[tr])
        proba = clf.predict_proba(X[te])[:, 1]
        pred = (proba >= THRESHOLD).astype(int)
        results.append({
            'condition': cond_id, 'label': cond_label, 'fold': fold,
            'auprc':    average_precision_score(y[te], proba),
            'auroc':    roc_auc_score(y[te], proba),
            'macro_f1': f1_score(y[te], pred, average='macro'),
            'bal_acc':  balanced_accuracy_score(y[te], pred),
            'mcc':      matthews_corrcoef(y[te], pred),
        })
        r = results[-1]
        print(f"  Fold {fold}: AUPRC={r['auprc']:.3f}, AUROC={r['auroc']:.3f}, "
              f"F1={r['macro_f1']:.3f}, MCC={r['mcc']:.3f}")

results = pd.DataFrame(results)
results.to_csv("../results/prostt5_mini_per_fold.csv", index=False)

print("\n=== ProstT5 mini-factorial aggregate (mean ± std over 5 folds) ===")
agg = results.groupby(['condition', 'label'])[['auprc','auroc','macro_f1','bal_acc','mcc']].agg(['mean','std']).round(3)
print(agg.to_string())

# Direct comparison with W7 ESM-2 disorder-pool baseline
print("\n=== ProstT5 vs ESM-2 (disorder-pool) headline comparison ===")
print("                          | Cond 1 (Seq only) | Cond 8 (Seq + all GO)")
print("ESM-2 disorder pool (W7): |     0.237         |     0.230")
prostt5_c1 = results[results['condition']==1]['auprc'].mean()
prostt5_c8 = results[results['condition']==8]['auprc'].mean()
print(f"ProstT5 disorder pool:    |     {prostt5_c1:.3f}         |     {prostt5_c8:.3f}")
print(f"ProstT5 - ESM-2:          |     {prostt5_c1-0.237:+.3f}         |     {prostt5_c8-0.230:+.3f}")

n=1279, positives=188, clusters=1168, threshold=0.147

=== Condition 1: Seq only (ProstT5) (X shape (1279, 1024)) ===
  Fold 0: AUPRC=0.208, AUROC=0.694, F1=0.458, MCC=0.237
  Fold 1: AUPRC=0.243, AUROC=0.657, F1=0.468, MCC=0.185
  Fold 2: AUPRC=0.184, AUROC=0.572, F1=0.403, MCC=0.060
  Fold 3: AUPRC=0.270, AUROC=0.648, F1=0.434, MCC=0.186
  Fold 4: AUPRC=0.309, AUROC=0.670, F1=0.472, MCC=0.163

=== Condition 8: Seq + all GO (ProstT5) (X shape (1279, 1149)) ===
  Fold 0: AUPRC=0.202, AUROC=0.679, F1=0.465, MCC=0.227
  Fold 1: AUPRC=0.245, AUROC=0.674, F1=0.485, MCC=0.216
  Fold 2: AUPRC=0.172, AUROC=0.572, F1=0.406, MCC=0.077
  Fold 3: AUPRC=0.255, AUROC=0.631, F1=0.425, MCC=0.177
  Fold 4: AUPRC=0.306, AUROC=0.682, F1=0.481, MCC=0.173

=== ProstT5 mini-factorial aggregate (mean ± std over 5 folds) ===
                                  auprc         auroc        macro_f1        bal_acc           mcc       
                                   mean    std   mean    std     mean    std    